<a href="https://colab.research.google.com/github/JaberAhmad555/flyrank-ml-internship/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JaberAhmad555/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring

**Task type:** **Ranking / scoring**

The decision is: **Which pages should a content editor review first?**

A simple yes/no classification is not enough because the editor cannot review every page at once. I want the model to give each page a priority score and rank the pages from highest to lowest priority. The top-ranked pages become a decision-support review queue.

A wrong high-priority call wastes editor time on a page that may not need attention. A wrong low-priority call may leave a genuinely declining page unreviewed.


In [1]:
# Setup: make sure the notebook can find the starter repo and dataset.
# This also works if the notebook was uploaded directly to Colab.

from pathlib import Path
import os
import subprocess

DATA_REL = Path("data/raw/content_refresh_anonymized.csv")
REPO_DIR = Path("/content/flyrank-ml-internship")

if not DATA_REL.exists():
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "-q",
             "https://github.com/JaberAhmad555/flyrank-ml-internship.git",
             str(REPO_DIR)],
            check=True
        )
    os.chdir(REPO_DIR)

print("Lane: Refresh / Content Opportunity Scoring")
print("Task type: Ranking / scoring")
print("Decision: Which pages should an editor review first?")
print("Dataset found:", DATA_REL.exists())


Lane: Refresh / Content Opportunity Scoring
Task type: Ranking / scoring
Decision: Which pages should an editor review first?
Dataset found: True


## 2. Target or proxy

My **proxy target** is `is_declining_label`.

- `1` = the page's observed `trend_direction` is **down**
- `0` = otherwise

For the ranking task, the model will estimate a decline-risk / priority score and use that score to order pages.

This is a **proxy**, not proof that refreshing a page will improve it. It only gives decision-support about which pages may deserve review first.

Important leakage rule: `trend_direction` and `trend_pct` help define the label, so I will **never use them as model features**.


In [2]:
import pandas as pd

df = pd.read_csv(DATA_REL)

# Build the proxy label from the observed trend direction.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

print("Rows:", len(df))
print("Declining pages:", int(df["is_declining_label"].sum()))
print("Declining rate:", f'{df["is_declining_label"].mean():.2%}')
print("Label values:", sorted(df["is_declining_label"].unique().tolist()))


Rows: 30000
Declining pages: 16262
Declining rate: 54.21%
Label values: [0, 1]


## 3. Success metric

My primary metric is **Precision@50**.

It asks:

> Among the 50 pages ranked highest for review, what fraction are actually labeled as declining?

This metric matches the real action because an editor has limited time and mainly cares whether the **top of the queue** contains useful pages.

I will call the result useful if **Precision@50 is at least 0.65 on a proper holdout set** and clearly beats the dataset's declining base rate. A Precision@50 of 0.65 means about 33 of the top 50 pages are truly declining.

I am choosing the metric before training so I do not change the definition of "good" after seeing model results.


In [3]:
base_rate = df["is_declining_label"].mean()
success_p50 = 0.65
k = 50

print(f"Declining base rate: {base_rate:.3f} ({base_rate:.2%})")
print(f"Random-like ranking would contain about {base_rate * k:.1f} declining pages in a top-{k} list.")
print(f"Success target: Precision@{k} >= {success_p50:.2f}")
print(f"That means about {success_p50 * k:.1f} declining pages in the top-{k} list.")


Declining base rate: 0.542 (54.21%)
Random-like ranking would contain about 27.1 declining pages in a top-50 list.
Success target: Precision@50 >= 0.65
That means about 32.5 declining pages in the top-50 list.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item / page.**

Each row contains signals about that page, such as recent impressions, CTR, average position, content age, days since the last update, and word count.

The output I eventually want is also one score per page, so this row-level grain matches the decision: **which page should be reviewed first?**

`content_id` and `client_id` are identifiers for grouping and splitting only; they are not model features.


In [4]:
# Show a real lane-focused slice of the dataframe.

display_cols = [
    "content_id",
    "client_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "is_declining_label",
]

lane_df = df[display_cols].copy()

print("Unit of analysis: one row = one content item / page")
print("Lane dataframe shape:", lane_df.shape)
display(lane_df.head(8))


Unit of analysis: one row = one content item / page
Lane dataframe shape: (30000, 9)


,content_id,client_id,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,word_count,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,0.76,10.6,187,20,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,15320,0.05,20.3,445,25,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,0.09,36.5,141,20,3515.0,1
3,content_331d6c4de07b,client_19581e27de,11751,0.49,6.2,463,22,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,0.13,44.0,263,14,2803.0,1
5,content_d4084a4bc775,client_f369cb89fc,3970,0.03,8.5,147,20,3080.0,1
6,content_9a34b442b552,client_8722616204,20,0.00,7.0,90,20,3059.0,1
7,content_a63219c6e95a,client_19581e27de,1724,0.06,21.2,445,22,NaN,0


## 5. Why ML beats a fixed rule here

A fixed rule such as:

> "Review the page if it has not been updated for 180+ days AND has 500+ impressions"

is easy to understand, but it is too rigid for this problem.

A page can decline for different combinations of signals. For example, age, recent visibility, CTR, average position, update recency, and content size may matter together. One hand-written threshold cannot easily capture all of those interactions.

ML can learn a **weighted combination of several safe signals** and create a smoother priority ranking. The output is still decision-support: an editor reviews the ranked pages before taking action.

I will compare ML against a simple rule baseline. ML only earns its place if it produces a more useful top-of-queue result on held-out data.


In [5]:
# Compare the size and coverage of one simple fixed-rule baseline.

simple_rule = (
    (df["days_since_last_update"] >= 180)
    & (df["impressions_90d"] >= 500)
)

rule_selected = df.loc[simple_rule]
total_declining = int(df["is_declining_label"].sum())
declining_caught = int(rule_selected["is_declining_label"].sum())

coverage = declining_caught / total_declining if total_declining else 0

print("Simple rule: days_since_last_update >= 180 AND impressions_90d >= 500")
print("Pages selected by rule:", len(rule_selected))
print("Declining pages selected:", declining_caught)
print("Total declining pages:", total_declining)
print("Share of all declining pages caught by this rule:", f"{coverage:.2%}")
print("\nInterpretation: one fixed rule can be very narrow, so a learned ranking may combine signals more flexibly.")


Simple rule: days_since_last_update >= 180 AND impressions_90d >= 500
Pages selected by rule: 17
Declining pages selected: 16
Total declining pages: 16262
Share of all declining pages caught by this rule: 0.10%

Interpretation: one fixed rule can be very narrow, so a learned ranking may combine signals more flexibly.


## Self-check

Before submitting:

- [x] Every section above is filled — markdown thinking AND code that backs it
- [x] The notebook runs top to bottom with no errors (tick this after **Runtime → Run all**)
- [x] No client names, URLs, or private queries are included in the analysis
- [x] Claims use careful words: observed, proxy, directional, decision-support
- [x] Committed to my repo under `work/notebooks/w02_ml_task_framing.ipynb` (tick this after saving to GitHub)

**Submission:** after both unchecked items are complete, submit the same public repository URL on the ML-03 card.
